# Notebook 05 — MySQL Analysis

1. Locates the project root and loads the `.env` file.
2. Reads the featured CSV and creates/tests the MySQL `engine` in the first code cell.
3. Normalizes column names and loads the data into `food_delivery_orders`.
4. Validates row counts and order status.
5. Runs 15 SQL analysis queries.

In [ ]:
from pathlib import Path
import os
import re
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "online_food_delivery_analysis_featured.csv"
ENV_PATH = PROJECT_ROOT / ".env"

# Load environment variables before reading credentials.
load_dotenv(dotenv_path=ENV_PATH, override=True)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        "Featured CSV not found. Run Notebook 03 first and confirm the file exists at: "
        f"{INPUT_PATH}"
    )

df = pd.read_csv(INPUT_PATH)
print("Project root:", PROJECT_ROOT)
print("Input file:", INPUT_PATH)
print("Input shape:", df.shape)
print("Input columns:", df.columns.tolist())
display(df.head())

# Read MySQL configuration.
MYSQL_HOST = os.getenv("MYSQL_HOST", "localhost")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE", "food_delivery_analysis")

print(".env file found:", ENV_PATH.exists())
print("MySQL username configured:", bool(MYSQL_USER))
print("MySQL password configured:", MYSQL_PASSWORD is not None)

if not MYSQL_USER or MYSQL_PASSWORD is None:
    raise ValueError(
        f"MySQL credentials are missing. Check MYSQL_USER and MYSQL_PASSWORD in: {ENV_PATH}"
    )

# Create the engine HERE so it exists before the data-load cell runs.
connection_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=MYSQL_DATABASE,
    query={"charset": "utf8mb4"},
)
engine = create_engine(connection_url, pool_pre_ping=True)

# Test the connection. The database must already exist.
with engine.connect() as connection:
    version = connection.execute(text("SELECT VERSION()")).scalar()
    print("MySQL connection successful!")
    print("MySQL version:", version)
    print("Database:", MYSQL_DATABASE)


Project root: d:\online food delivery analysis final
Input file: d:\online food delivery analysis final\data\processed\online_food_delivery_analysis_featured.csv
Input shape: (100000, 42)
Input columns: ['Order_ID', 'Customer_ID', 'Customer_Age', 'Customer_Gender', 'City', 'Area', 'Restaurant_ID', 'Restaurant_Name', 'Cuisine_Type', 'Order_Date', 'Order_Time', 'Delivery_Time_Min', 'Distance_km', 'Order_Value', 'Discount_Applied', 'Final_Amount', 'Payment_Mode', 'Order_Status', 'Cancellation_Reason', 'Delivery_Partner_ID', 'Delivery_Rating', 'Restaurant_Rating', 'Order_Day', 'Peak_Hour', 'Profit_Margin', 'Order_Time_Parsed', 'Order_Year', 'Order_Month', 'Order_Month_Name', 'Order_Weekday_Name', 'Order_Hour', 'Is_Weekend', 'Peak_Period', 'Peak_Hour_Recalculated', 'Customer_Age_Group', 'Delivery_Performance', 'Discount_Percentage', 'Profit_Margin_Percentage', 'Order_Value_Band', 'Is_Cancelled', 'Is_Delivered', 'Is_Non_Delivered']


,Order_ID,Customer_ID,Customer_Age,Customer_Gender,City,Area,Restaurant_ID,Restaurant_Name,Cuisine_Type,Order_Date,...,Peak_Period,Peak_Hour_Recalculated,Customer_Age_Group,Delivery_Performance,Discount_Percentage,Profit_Margin_Percentage,Order_Value_Band,Is_Cancelled,Is_Delivered,Is_Non_Delivered
0,ORD000001,CUST03987,24.0,Male,Chennai,OMR,RES1298,Restaurant_1298,Fast Food,2024-06-05,...,Off-Peak,False,18–25,Slow,5.259836,2.9,"1,001–2,000",0,1,0
1,ORD000002,CUST01917,45.0,Female,Hyderabad,NaN,RES1191,Restaurant_1191,Chinese,2024-05-11,...,Off-Peak,False,36–45,Standard,4.083466,21.4,Up to 500,0,1,0
2,ORD000003,CUST13338,57.0,Male,Bengaluru,Indiranagar,RES0681,Restaurant_0681,Indian,2024-02-02,...,Off-Peak,False,46–60,Slow,0.000000,20.4,Up to 500,0,1,0
3,ORD000004,CUST09787,66.0,Female,Chennai,Velachery,RES0453,Restaurant_0453,Indian,2024-03-27,...,Evening Peak,True,Over 60,Standard,6.838261,29.3,"501–1,000",0,1,0
4,ORD000005,CUST15793,43.0,Female,Chennai,Anna Nagar,RES0268,Restaurant_0268,Indian,2024-07-11,...,Morning Peak,True,36–45,NaN,17.486492,NaN,"501–1,000",0,0,1


.env file found: True
MySQL username configured: True
MySQL password configured: True
MySQL connection successful!
MySQL version: 8.0.45
Database: food_delivery_analysis


## 1. Configure MySQL credentials safely

In [ ]:

if "engine" not in globals():
    raise NameError("MySQL engine is not defined. Run the first setup code cell (Cell 2) before continuing.")

print("MySQL engine is ready.")
print("Target database:", MYSQL_DATABASE)


MySQL engine is ready.
Target database: food_delivery_analysis


## 2. Prepare column names and load the table

In [3]:
import re

def to_sql_name(name):
    name = re.sub(r"[^0-9a-zA-Z]+", "_", str(name).strip())
    name = re.sub(r"_+", "_", name).strip("_").lower()
    if not name:
        name = "column"
    if name[0].isdigit():
        name = f"col_{name}"
    return name

sql_df = df.copy()
sql_df.columns = [to_sql_name(column) for column in sql_df.columns]

# Convert boolean-like columns to nullable integer 0/1 values for MySQL compatibility.
for column in sql_df.columns:
    if str(sql_df[column].dtype) in ("bool", "boolean"):
        sql_df[column] = sql_df[column].map({True: 1, False: 0})

# Convert common date columns to ISO dates where present.
for column in ["order_date"]:
    if column in sql_df.columns:
        sql_df[column] = pd.to_datetime(sql_df[column], errors="coerce").dt.date

TABLE_NAME = "food_delivery_orders"

sql_df.to_sql(
    TABLE_NAME,
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=500,
    method="multi"
)

with engine.connect() as connection:
    table_count = connection.execute(text(f"SELECT COUNT(*) FROM `{TABLE_NAME}`")).scalar()

print("Loaded table:", TABLE_NAME)
print("Rows in MySQL:", table_count)
print("Columns in SQL table:", len(sql_df.columns))
print("SQL columns:", sql_df.columns.tolist())

if table_count != len(sql_df):
    raise AssertionError(f"Loaded row count {table_count} does not match CSV row count {len(sql_df)}")


Loaded table: food_delivery_orders
Rows in MySQL: 100000
Columns in SQL table: 42
SQL columns: ['order_id', 'customer_id', 'customer_age', 'customer_gender', 'city', 'area', 'restaurant_id', 'restaurant_name', 'cuisine_type', 'order_date', 'order_time', 'delivery_time_min', 'distance_km', 'order_value', 'discount_applied', 'final_amount', 'payment_mode', 'order_status', 'cancellation_reason', 'delivery_partner_id', 'delivery_rating', 'restaurant_rating', 'order_day', 'peak_hour', 'profit_margin', 'order_time_parsed', 'order_year', 'order_month', 'order_month_name', 'order_weekday_name', 'order_hour', 'is_weekend', 'peak_period', 'peak_hour_recalculated', 'customer_age_group', 'delivery_performance', 'discount_percentage', 'profit_margin_percentage', 'order_value_band', 'is_cancelled', 'is_delivered', 'is_non_delivered']


## 3. Verify the loaded table

In [4]:
with engine.connect() as connection:
    display(pd.read_sql(text(f"SELECT order_status, COUNT(*) AS order_count FROM `{TABLE_NAME}` GROUP BY order_status ORDER BY order_count DESC"), connection))
    display(pd.read_sql(text(f"SELECT COUNT(*) AS total_rows, COUNT(DISTINCT order_id) AS distinct_orders FROM `{TABLE_NAME}`"), connection))
    display(pd.read_sql(text(f"SELECT COUNT(*) AS delivered_orders FROM `{TABLE_NAME}` WHERE LOWER(order_status) = 'delivered'"), connection))


,order_status,order_count
0,Delivered,89288
1,Cancelled,6143
2,Out for Delivery,2541
3,Preparing,2028


,total_rows,distinct_orders
0,100000,100000


,delivered_orders
0,89288


## 4. SQL business analysis — 15 queries


### Query 1 — Top customers by total spending

In [5]:
q1 = f"""
SELECT customer_id,
       COUNT(*) AS delivered_orders_with_amount,
       ROUND(SUM(final_amount), 2) AS total_spending
FROM `{TABLE_NAME}`
WHERE LOWER(order_status) = 'delivered'
  AND final_amount IS NOT NULL
GROUP BY customer_id
ORDER BY total_spending DESC
LIMIT 10
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q1), connection))


,customer_id,delivered_orders_with_amount,total_spending
0,CUST00466,14,19050.92
1,CUST16895,13,17999.84
2,CUST09468,13,17909.74
3,CUST13262,10,17759.44
4,CUST13051,11,17677.01
5,CUST16981,13,17548.37
6,CUST10853,13,16949.35
7,CUST00343,12,16900.47
8,CUST15981,13,16888.60
9,CUST06664,14,16718.54


### Query 2 — Customer demographics by city and gender

In [6]:
q2 = f"""
SELECT city, customer_gender,
       COUNT(*) AS total_orders,
       COUNT(DISTINCT customer_id) AS unique_customers
FROM `{TABLE_NAME}`
GROUP BY city, customer_gender
ORDER BY total_orders DESC
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q2), connection))


,city,customer_gender,total_orders,unique_customers
0,Chennai,Female,10256,7796
1,Chennai,Male,10088,7702
2,Bengaluru,Male,8384,6661
3,Bengaluru,Female,8300,6676
4,Mumbai,Female,7959,6415
5,Mumbai,Male,7790,6340
6,Hyderabad,Female,6021,5159
7,Hyderabad,Male,5870,5035
8,Delhi,Female,5679,4861
9,Delhi,Male,5626,4894


### Query 3 — Order volume by month and weekday

In [7]:
q3 = f"""
SELECT DATE_FORMAT(order_date, '%Y-%m') AS order_month,
       order_day,
       COUNT(*) AS total_orders
FROM `{TABLE_NAME}`
GROUP BY DATE_FORMAT(order_date, '%Y-%m'), order_day
ORDER BY order_month, order_day
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q3), connection))


,order_month,order_day,total_orders
0,2024-01,Weekday,6217
1,2024-01,Weekend,2192
2,2024-02,Weekday,5758
3,2024-02,Weekend,2193
4,2024-03,Weekday,5751
5,2024-03,Weekend,2734
6,2024-04,Weekday,6004
7,2024-04,Weekend,2191
8,2024-05,Weekday,6242
9,2024-05,Weekend,2123


### Query 4 — Revenue and average order value by city

In [8]:
q4 = f"""
SELECT city,
       COUNT(*) AS delivered_orders_with_amount,
       ROUND(SUM(final_amount), 2) AS recorded_revenue,
       ROUND(AVG(final_amount), 2) AS average_final_amount
FROM `{TABLE_NAME}`
WHERE LOWER(order_status) = 'delivered'
  AND final_amount IS NOT NULL
GROUP BY city
ORDER BY recorded_revenue DESC
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q4), connection))


,city,delivered_orders_with_amount,recorded_revenue,average_final_amount
0,Chennai,19397,19604968.04,1010.72
1,Bengaluru,15750,15952450.64,1012.85
2,Mumbai,14935,15010169.56,1005.03
3,Hyderabad,11278,11365309.33,1007.74
4,Delhi,10702,10769573.05,1006.31
5,Pune,8792,8786219.33,999.34
6,Kolkata,7073,7197296.83,1017.57
7,NaN,484,472177.72,975.57


### Query 5 — Discount and profit margin by city

Profit margin is stored as a ratio; multiply by 100 to display percentage.

In [9]:
q5 = f"""
SELECT city,
       ROUND(AVG(discount_applied), 2) AS average_discount,
       ROUND(AVG(profit_margin) * 100, 2) AS average_profit_margin_pct,
       COUNT(profit_margin) AS orders_with_profit_margin
FROM `{TABLE_NAME}`
GROUP BY city
ORDER BY average_profit_margin_pct DESC
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q5), connection))


,city,average_discount,average_profit_margin_pct,orders_with_profit_margin
0,Hyderabad,75.68,17.72,11381
1,NaN,72.48,17.70,490
2,Kolkata,74.83,17.68,7139
3,Mumbai,74.99,17.66,15087
4,Bengaluru,77.66,17.65,15925
5,Chennai,76.14,17.63,19582
6,Pune,76.50,17.60,8874
7,Delhi,76.28,17.52,10810


### Query 6 — Revenue and order volume by cuisine

In [10]:
q6 = f"""
SELECT cuisine_type,
       COUNT(*) AS delivered_orders_with_amount,
       ROUND(SUM(final_amount), 2) AS recorded_revenue,
       ROUND(AVG(final_amount), 2) AS average_final_amount
FROM `{TABLE_NAME}`
WHERE LOWER(order_status) = 'delivered'
  AND final_amount IS NOT NULL
GROUP BY cuisine_type
ORDER BY recorded_revenue DESC
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q6), connection))


,cuisine_type,delivered_orders_with_amount,recorded_revenue,average_final_amount
0,Indian,17757,17922156.71,1009.30
1,South Indian,14054,14260888.30,1014.72
2,Fast Food,13392,13610330.93,1016.30
3,Chinese,10641,10663679.99,1002.13
4,Italian,7083,7144852.83,1008.73
5,Arabian,5314,5347260.52,1006.26
6,Thai,5302,5318594.47,1003.13
7,Healthy,5211,5267730.45,1010.89
8,Desserts,5248,5161777.60,983.57
9,Mexican,4409,4460892.70,1011.77


### Query 7 — Average delivery time by distance band

In [11]:
q7 = f"""
SELECT CASE
         WHEN distance_km <= 2 THEN '0-2 km'
         WHEN distance_km <= 5 THEN '>2-5 km'
         WHEN distance_km <= 10 THEN '>5-10 km'
         WHEN distance_km > 10 THEN '>10 km'
         ELSE 'Unknown'
       END AS distance_band,
       COUNT(*) AS delivered_orders_with_distance_and_time,
       ROUND(AVG(delivery_time_min), 2) AS average_delivery_time_min
FROM `{TABLE_NAME}`
WHERE LOWER(order_status) = 'delivered'
  AND delivery_time_min IS NOT NULL
  AND distance_km IS NOT NULL
GROUP BY distance_band
ORDER BY average_delivery_time_min
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q7), connection))


,distance_band,delivered_orders_with_distance_and_time,average_delivery_time_min
0,0-2 km,4568,40.88
1,>2-5 km,21289,46.86
2,>5-10 km,32638,57.17
3,>10 km,29604,79.95


### Query 8 — Delivery delay categories

Thresholds are project assumptions: Fast ≤30 minutes, Standard 31–60 minutes, Slow >60 minutes.

In [12]:

q8 = """
SELECT
    delivery_performance,
    COUNT(*) AS delivered_orders
FROM (
    SELECT
        CASE
            WHEN delivery_time_min <= 30 THEN 'Fast'
            WHEN delivery_time_min <= 60 THEN 'Standard'
            ELSE 'Slow'
        END AS delivery_performance
    FROM food_delivery_orders
    WHERE LOWER(order_status) = 'delivered'
      AND delivery_time_min IS NOT NULL
) AS delivery_categories
GROUP BY delivery_performance
ORDER BY delivered_orders DESC
"""

with engine.connect() as connection:
    display(pd.read_sql(text(q8), connection))


,delivery_performance,delivered_orders
0,Standard,46759
1,Slow,40700
2,Fast,1829


### Query 9 — Ratings by cuisine

In [13]:
q9 = f"""
SELECT cuisine_type,
       COUNT(delivery_rating) AS orders_with_delivery_rating,
       ROUND(AVG(delivery_rating), 2) AS average_delivery_rating,
       COUNT(restaurant_rating) AS orders_with_restaurant_rating,
       ROUND(AVG(restaurant_rating), 2) AS average_restaurant_rating
FROM `{TABLE_NAME}`
WHERE LOWER(order_status) = 'delivered'
GROUP BY cuisine_type
ORDER BY average_delivery_rating DESC
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q9), connection))


,cuisine_type,orders_with_delivery_rating,average_delivery_rating,orders_with_restaurant_rating,average_restaurant_rating
0,Fast Food,12971,4.20,13161,4.09
1,Thai,5128,4.20,5209,4.10
2,Arabian,5153,4.20,5231,4.09
3,Mexican,4253,4.19,4344,4.09
4,Italian,6844,4.19,6966,4.09
5,Healthy,5053,4.18,5109,4.09
6,Desserts,5078,4.18,5169,4.09
7,Indian,17205,4.16,17466,4.09
8,South Indian,13636,4.16,13840,4.08
9,Chinese,10300,4.15,10475,4.08


### Query 10 — Top restaurants by recorded revenue

In [14]:
q10 = f"""
SELECT restaurant_id, restaurant_name,
       COUNT(*) AS delivered_orders_with_amount,
       ROUND(SUM(final_amount), 2) AS recorded_revenue,
       ROUND(AVG(restaurant_rating), 2) AS average_restaurant_rating
FROM `{TABLE_NAME}`
WHERE LOWER(order_status) = 'delivered'
  AND final_amount IS NOT NULL
GROUP BY restaurant_id, restaurant_name
ORDER BY recorded_revenue DESC
LIMIT 10
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q10), connection))


,restaurant_id,restaurant_name,delivered_orders_with_amount,recorded_revenue,average_restaurant_rating
0,RES0542,Restaurant_0542,86,90246.65,4.07
1,RES0280,Restaurant_0280,74,89965.98,4.06
2,RES0590,Restaurant_0590,73,89200.07,3.95
3,RES1252,Restaurant_1252,78,88183.79,4.11
4,RES1195,Restaurant_1195,86,88166.00,4.04
5,RES0561,Restaurant_0561,83,86751.24,4.02
6,RES0271,Restaurant_0271,82,86747.62,4.07
7,RES0760,Restaurant_0760,75,85370.35,4.02
8,RES1177,Restaurant_1177,78,84349.86,4.07
9,RES0215,Restaurant_0215,76,84329.84,4.02


### Query 11 — Cancellation rate by city

In [15]:
q11 = f"""
SELECT city,
       COUNT(*) AS total_orders,
       SUM(CASE WHEN LOWER(order_status) = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
       ROUND(100.0 * SUM(CASE WHEN LOWER(order_status) = 'cancelled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancellation_rate_pct
FROM `{TABLE_NAME}`
GROUP BY city
ORDER BY cancellation_rate_pct DESC
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q11), connection))


,city,total_orders,cancelled_orders,cancellation_rate_pct
0,NaN,548,38.0,6.93
1,Delhi,12150,786.0,6.47
2,Bengaluru,17906,1145.0,6.39
3,Mumbai,16889,1049.0,6.21
4,Hyderabad,12777,784.0,6.14
5,Chennai,21880,1314.0,6.01
6,Pune,9909,572.0,5.77
7,Kolkata,7941,455.0,5.73


### Query 12 — Cancellation reasons

In [16]:
q12 = f"""
SELECT cancellation_reason,
       COUNT(*) AS cancelled_orders
FROM `{TABLE_NAME}`
WHERE LOWER(order_status) = 'cancelled'
GROUP BY cancellation_reason
ORDER BY cancelled_orders DESC
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q12), connection))


,cancellation_reason,cancelled_orders
0,Restaurant Issue,1710
1,Customer Cancelled,1634
2,Rider Unavailable,1212
3,Incorrect Address,849
4,Payment Issue,738


### Query 13 — Peak-hour order and revenue analysis

In [17]:
q13 = f"""
SELECT peak_period,
       COUNT(*) AS total_orders,
       SUM(CASE WHEN LOWER(order_status) = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
       ROUND(SUM(CASE WHEN LOWER(order_status) = 'delivered' THEN final_amount ELSE 0 END), 2) AS recorded_revenue
FROM `{TABLE_NAME}`
GROUP BY peak_period
ORDER BY total_orders DESC
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q13), connection))


,peak_period,total_orders,delivered_orders,recorded_revenue
0,Off-Peak,62872,56185.0,56270124.91
1,Evening Peak,16359,14587.0,14470363.48
2,Morning Peak,12484,11139.0,11051927.34
3,Lunch Peak,8285,7377.0,7365748.77


### Query 14 — Payment mode analysis

In [18]:
q14 = f"""
SELECT payment_mode,
       COUNT(*) AS total_orders,
       SUM(CASE WHEN LOWER(order_status) = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
       SUM(CASE WHEN LOWER(order_status) = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
       ROUND(AVG(CASE WHEN LOWER(order_status) = 'delivered' THEN final_amount END), 2) AS average_delivered_final_amount
FROM `{TABLE_NAME}`
GROUP BY payment_mode
ORDER BY total_orders DESC
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q14), connection))


,payment_mode,total_orders,delivered_orders,cancelled_orders,average_delivered_final_amount
0,UPI,41689,37223.0,2567.0,1009.62
1,Credit Card,22023,19685.0,1342.0,1010.35
2,Debit Card,15882,14153.0,971.0,1005.97
3,COD,11901,10628.0,751.0,1001.74
4,Wallet,8014,7159.0,482.0,1014.45
5,NaN,491,440.0,30.0,969.59


### Query 15 — Order value, discount, and margin by age group

In [19]:
q15 = f"""
SELECT customer_age_group,
       COUNT(*) AS total_orders,
       ROUND(AVG(order_value), 2) AS average_order_value,
       ROUND(AVG(discount_applied), 2) AS average_discount,
       ROUND(AVG(profit_margin) * 100, 2) AS average_profit_margin_pct
FROM `{TABLE_NAME}`
GROUP BY customer_age_group
ORDER BY total_orders DESC
"""
with engine.connect() as connection:
    display(pd.read_sql(text(q15), connection))


,customer_age_group,total_orders,average_order_value,average_discount,average_profit_margin_pct
0,46–60,27937,1085.38,76.76,17.60
1,26–35,18683,1087.51,76.96,17.65
2,36–45,18499,1084.43,75.69,17.67
3,Over 60,18462,1085.85,74.78,17.72
4,18–25,14943,1081.33,76.09,17.57
5,NaN,1476,1080.73,73.75,17.65


## 5. Save SQL results 

In [20]:
RESULTS_DIR = PROJECT_ROOT / "outputs" / "sql"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

query_map = {
    "top_customers.csv": q1,
    "revenue_by_city.csv": q4,
    "cuisine_performance.csv": q6,
    "delivery_time_by_distance.csv": q7,
    "top_restaurants.csv": q10,
    "cancellation_rate_by_city.csv": q11,
    "cancellation_reasons.csv": q12,
    "payment_mode_analysis.csv": q14,
}

with engine.connect() as connection:
    for filename, query in query_map.items():
        result_df = pd.read_sql(text(query), connection)
        result_df.to_csv(RESULTS_DIR / filename, index=False)
        print(f"Saved {filename}: {len(result_df)} rows")

print("SQL result files saved to:", RESULTS_DIR.resolve())


Saved top_customers.csv: 10 rows
Saved revenue_by_city.csv: 8 rows
Saved cuisine_performance.csv: 10 rows
Saved delivery_time_by_distance.csv: 4 rows
Saved top_restaurants.csv: 10 rows
Saved cancellation_rate_by_city.csv: 8 rows
Saved cancellation_reasons.csv: 5 rows
Saved payment_mode_analysis.csv: 6 rows
SQL result files saved to: D:\online food delivery analysis final\outputs\sql
